# 2.3a Evaluate the fraud model

**Story:** The fraud model is trained. Before it goes near production, you measure how good it is,
record experiments so they can be repeated, create a baseline for later drift checks, explain its
decisions to the fraud team, and learn to debug a training run that does not converge.

**Task 2.3:** Analyze and evaluate the performance of ML and AI systems (traditional ML part).

| Skill | Topic | Status |
|---|---|---|
| 2.3.6 | Evaluation techniques for traditional ML (classification, regression) | ✅ |
| 2.3.1 | Reproducible experiments with MLflow on SageMaker AI | ✅ |
| 2.3.2 | Performance baselines and drift detection (baseline here, drift run in 4.1a) | ✅ |
| 2.3.3 | Shadow variant vs production variant | 📝 (hands-on in 4.1a) |
| 2.3.4 | Explain model outputs (SHAP, SageMaker Clarify) | ✅ |
| 2.3.5 | Debug model convergence issues (learning rate, SageMaker Debugger) | ✅ |

💰 One batch transform job, two processing jobs, one training job (cents each) and an MLflow tracking
server (about USD 0.60 per hour while it exists; deleted in Cleanup). Run time about 60 minutes,
mostly waiting.

## Key points for the exam

- **Imbalanced classes** (2-5% fraud): accuracy is misleading. Use precision, recall, F1, **PR AUC**, and choose a threshold from business cost.
- **Precision** = of the claims we flagged, how many were fraud. **Recall** = of all fraud, how many we caught.
  Lower threshold → recall up, precision down.
- **ROC AUC** is threshold-free and compares rankings. **RMSE** punishes large errors more than **MAE**. **R²** = share of variance explained.
- **MLflow on SageMaker AI** = managed tracking server: log parameters, metrics and model artifacts per run; compare runs; register models.
- **Model Monitor baseline** = statistics + constraints computed from the training data. Later, captured production data is compared with it.
- Four monitor types: **data quality**, **model quality** (needs ground truth labels), **bias drift**, **feature attribution drift** (last two use Clarify).
- **SageMaker Clarify** explains predictions with **SHAP** (global and per prediction) and measures bias.
- **SageMaker Debugger** built-in rules watch a training job: `loss_not_decreasing`, `overfit`, `overtraining`, `vanishing_gradient`, `exploding_tensor`.

| Question | Metric |
|---|---|
| Missing a fraud case is very costly | Recall (and a lower threshold) |
| Investigators' time is limited | Precision |
| One number for a rare positive class | F1 or PR AUC |
| Regression with a few huge errors that matter | RMSE |
| Regression, robust to outliers | MAE |

## Setup

In [ ]:
import sys; sys.path.append("..")
import json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from botocore.exceptions import ClientError
from common import utils as U, catchup as C, features as F
warnings.filterwarnings("ignore")
print(U.REGION, U.bucket())

Catch-up: training data and the baseline fraud model (trained in 2.2a; built now if missing, about 5 minutes).

In [ ]:
state = C.ensure_training_data()
MODEL_DATA = C.ensure_fraud_model()
print(MODEL_DATA)

SageMaker session and client, and the three data splits in pandas (label is the first column).

In [ ]:
import sagemaker
sm_session = sagemaker.Session(boto_session=U.session, default_bucket=U.bucket())
sm = U.client("sagemaker")
cols = [F.LABEL] + F.FEATURES
train, val, test = (U.read_csv(f"{C.PROCESSED}/{n}/{n}.csv", header=None, names=cols)
                    for n in ("train", "validation", "test"))
Xtr, ytr, Xva, yva = train[F.FEATURES], train[F.LABEL], val[F.FEATURES], val[F.LABEL]
print(train.shape, val.shape, test.shape)

## 2.3.1 (first step) Start the MLflow tracking server now

Creating the server takes about 20 to 25 minutes, so we start it here and use it later in this notebook.
💰 A `Small` server costs about USD 0.60 per hour until it is deleted (Cleanup section).

In [ ]:
MLFLOW_NAME = "safedrive-mlflow"
try:
    status = sm.describe_mlflow_tracking_server(TrackingServerName=MLFLOW_NAME)["TrackingServerStatus"]
    print("already exists:", status)
except ClientError:
    r = sm.create_mlflow_tracking_server(
        TrackingServerName=MLFLOW_NAME, ArtifactStoreUri=U.s3_uri(f"{U.PREFIX}/mlflow"),
        TrackingServerSize="Small", RoleArn=U.role_arn(), Tags=U.TAGS)
    print("creating:", r["TrackingServerArn"])

## 2.3.6 Evaluate a classifier: score the test set

- Evaluate on data the model has **never seen** (the test split), not on training data.
- We score the test set with a **batch transform** job: no endpoint needed, works for any container.
- `input_filter` drops the label column before inference; `join_source` + `output_filter` put the
  label and the prediction side by side in the output.

Create a SageMaker **model** (container image + artifact). It is reused by the Clarify job later.

In [ ]:
EVAL_MODEL = "safedrive-fraud-eval"
try:
    sm.describe_model(ModelName=EVAL_MODEL)
    print("model exists")
except ClientError:
    sm.create_model(ModelName=EVAL_MODEL, ExecutionRoleArn=U.role_arn(), Tags=U.TAGS,
                    PrimaryContainer={"Image": U.get_state("xgb_image"), "ModelDataUrl": MODEL_DATA})
    print("created", EVAL_MODEL)

Run the batch transform on the test file (about 5 minutes). The test CSV has the label in column 0,
so `$[1:]` sends only the features, and `$[0,-1]` keeps the label and the score.

In [ ]:
from sagemaker.transformer import Transformer
BATCH_OUT = f"{U.PREFIX}/evaluation/batch"
transformer = Transformer(model_name=EVAL_MODEL, instance_count=1, instance_type="ml.m5.large",
                          output_path=U.s3_uri(BATCH_OUT), accept="text/csv", assemble_with="Line",
                          sagemaker_session=sm_session, tags=U.TAGS, base_transform_job_name="safedrive-fraud-eval")
transformer.transform(state["test_s3"], content_type="text/csv", split_type="Line",
                      input_filter="$[1:]", join_source="Input", output_filter="$[0,-1]", wait=True, logs=False)
print(transformer.latest_transform_job.name)

Read the output: one row per test claim with the true label and the fraud score.

In [ ]:
scored = U.read_csv(f"{BATCH_OUT}/test.csv.out", header=None, names=["is_fraud", "score"])
y_true, y_score = scored["is_fraud"].astype(int).values, scored["score"].values
print(len(scored), "claims, fraud rate", round(y_true.mean(), 4))
scored.describe().T

### Confusion matrix, precision, recall, F1 at threshold 0.5
Rows are the truth, columns the prediction. Read the four cells: TN, FP / FN, TP.

In [ ]:
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score, accuracy_score,
                             roc_auc_score, average_precision_score, precision_recall_curve, roc_curve)
pred = (y_score >= 0.5).astype(int)
cm = pd.DataFrame(confusion_matrix(y_true, pred), index=["true legit", "true fraud"],
                  columns=["pred legit", "pred fraud"])
print("precision", round(precision_score(y_true, pred), 3), "| recall", round(recall_score(y_true, pred), 3),
      "| F1", round(f1_score(y_true, pred), 3), "| accuracy", round(accuracy_score(y_true, pred), 3))
cm

Why accuracy misleads: a "model" that says *never fraud* gets high accuracy and catches nothing.

In [ ]:
never = np.zeros_like(y_true)
print("accuracy of 'never fraud':", round(accuracy_score(y_true, never), 3),
      "| recall:", recall_score(y_true, never))

### Threshold-free metrics: ROC AUC and PR AUC
ROC AUC: chance = 0.5. PR AUC (average precision): chance = the fraud rate, so compare it with that.

In [ ]:
print("ROC AUC:", round(roc_auc_score(y_true, y_score), 4))
print("PR AUC :", round(average_precision_score(y_true, y_score), 4), "(baseline =", round(y_true.mean(), 4), ")")

Plot both curves. For rare classes the PR curve shows much more clearly how hard the problem is.

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_score)
prec, rec, _ = precision_recall_curve(y_true, y_score)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], "--"); ax[0].set(title="ROC", xlabel="false positive rate", ylabel="recall")
ax[1].plot(rec, prec); ax[1].axhline(y_true.mean(), ls="--"); ax[1].set(title="Precision-recall", xlabel="recall", ylabel="precision")
plt.tight_layout(); plt.show()

### Choose the threshold from business cost
Assume a missed fraud costs USD 2,500 (the payout) and a false alarm costs USD 400 (an investigator's
time). Compute the total cost at several thresholds and look for the minimum.

In [ ]:
COST_FN, COST_FP = 2500, 400
rows = []
for t in np.arange(0.1, 0.95, 0.1):
    p = (y_score >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, p).ravel()
    rows.append({"threshold": round(t, 2), "flagged": int(tp + fp), "precision": precision_score(y_true, p, zero_division=0),
                 "recall": recall_score(y_true, p), "f1": f1_score(y_true, p), "cost_usd": fn * COST_FN + fp * COST_FP})
sweep = pd.DataFrame(rows)
print("cheapest threshold:", sweep.loc[sweep["cost_usd"].idxmin(), "threshold"])
sweep.round(3)

**More techniques to know:** k-fold cross-validation (more stable estimate on small data), stratified
splits for imbalanced labels, a time-based split when data has a time order, and a final untouched test set.

### Regression metrics: RMSE, MAE, R²
Two quick local regressors. `annual_premium` is easy to predict from the policy data. `claim_amount`
is mostly random in our synthetic data, so it shows what "no better than predicting the mean" looks like.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
claims = U.read_csv(f"{C.RAW}/claims.csv").merge(U.read_csv(f"{C.RAW}/policies.csv"), on="policy_id")
X = pd.get_dummies(claims[["vehicle_value", "vehicle_year", "age", "num_vehicles", "claim_type"]], dtype=float)
print(X.shape)

Train one small model per target and compare with the simplest baseline (always predict the mean).

In [ ]:
def regression_report(target):
    Xa, Xb, ya, yb = train_test_split(X, claims[target], test_size=0.2, random_state=42)
    p = xgb.XGBRegressor(n_estimators=60, max_depth=3, learning_rate=0.1).fit(Xa, ya).predict(Xb)
    return {"rmse": np.sqrt(mean_squared_error(yb, p)), "mae": mean_absolute_error(yb, p), "r2": r2_score(yb, p),
            "rmse_of_mean_baseline": np.sqrt(mean_squared_error(yb, np.full(len(yb), ya.mean())))}

pd.DataFrame({t: regression_report(t) for t in ("annual_premium", "claim_amount")}).T.round(3)

- R² near 1: the features explain the target. R² near 0 (or negative): the model is no better than the mean.
- RMSE is larger than MAE when some errors are very large (RMSE squares them).

## 2.3.1 Reproducible experiments with MLflow on SageMaker AI

- A **run** stores parameters, metrics, tags and artifacts (the model). Runs are grouped in an **experiment**.
- To make a result repeatable, log: code version, **data version** (S3 URI), hyperparameters, random seed, library versions.
- The tracking server is managed by SageMaker; the `sagemaker-mlflow` plugin lets the `mlflow` client use the server ARN as its tracking URI.
- MLflow models can be registered into **SageMaker Model Registry** automatically.

Wait until the tracking server (started at the top of this notebook) is ready.

In [ ]:
U.wait_until(lambda: sm.describe_mlflow_tracking_server(TrackingServerName=MLFLOW_NAME)["TrackingServerStatus"],
             ok=("Created", "Started", "Updated"), bad=("CreateFailed", "StartFailed"), delay=60,
             timeout=2700, label="mlflow server")

Point the MLflow client at the server and choose an experiment.

In [ ]:
import mlflow
server = sm.describe_mlflow_tracking_server(TrackingServerName=MLFLOW_NAME)
mlflow.set_tracking_uri(server["TrackingServerArn"])
mlflow.set_experiment("safedrive-fraud")
print("MLflow", server["MlflowVersion"], "| client", mlflow.__version__)

A function that does one tracked run: train locally with a fixed seed, then log parameters, metrics,
the data version and the model.

In [ ]:
def tracked_run(name, params):
    with mlflow.start_run(run_name=name) as run:
        model = xgb.XGBClassifier(**params, random_state=42).fit(Xtr, ytr)
        p = model.predict_proba(Xva)[:, 1]
        mlflow.log_params({**params, "seed": 42})
        mlflow.set_tag("train_data", state["train_s3"])
        mlflow.log_metrics({"val_auc": roc_auc_score(yva, p), "val_pr_auc": average_precision_score(yva, p)})
        try:
            mlflow.xgboost.log_model(model, "model")
        except Exception as e:  # client and server MLflow versions can differ
            print("model not logged:", str(e)[:120])
    return run.info.run_id

print("ready")

Run two configurations: shallow trees and deep trees.

In [ ]:
run_a = tracked_run("shallow", {"n_estimators": 150, "max_depth": 3, "learning_rate": 0.1})
run_b = tracked_run("deep", {"n_estimators": 150, "max_depth": 8, "learning_rate": 0.1})
print(run_a, run_b)

Compare the runs. The same table (with charts) is in the MLflow UI.

In [ ]:
runs = mlflow.search_runs(experiment_names=["safedrive-fraud"], order_by=["metrics.val_auc DESC"])
runs[["tags.mlflow.runName", "params.max_depth", "params.learning_rate", "metrics.val_auc", "metrics.val_pr_auc"]]

Optional: a temporary link to open the MLflow UI in your browser.

In [ ]:
url = sm.create_presigned_mlflow_tracking_server_url(TrackingServerName=MLFLOW_NAME)["AuthorizedUrl"]
print(url[:120], "...")

## 2.3.2 Performance baselines and drift detection

- **Baseline** = what "normal" looks like: per-feature statistics and constraints from the training data,
  plus the model's metrics on the test set (the ROC AUC and PR AUC above are the *model quality* baseline).
- **Drift detection** = compare production data or predictions with the baseline on a schedule and alarm on violations.
- **Data drift**: input distribution changes. **Concept drift**: the relation between inputs and label changes
  (needs ground truth to detect, through the model quality monitor).

Start a Model Monitor **baseline job** on the baseline file (training features with a header).
It runs as a processing job and takes about 8 minutes; we do not block here.

In [ ]:
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat
BASELINE_PREFIX = f"{U.PREFIX}/monitoring/baseline"
monitor = DefaultModelMonitor(role=U.role_arn(), instance_count=1, instance_type="ml.m5.large",
                              volume_size_in_gb=10, max_runtime_in_seconds=1800,
                              sagemaker_session=sm_session, tags=U.TAGS, base_job_name="safedrive-baseline")
monitor.suggest_baseline(baseline_dataset=state["baseline_s3"], dataset_format=DatasetFormat.csv(header=True),
                         output_s3_uri=U.s3_uri(BASELINE_PREFIX), wait=False, logs=False)
BASELINE_JOB = monitor.latest_baselining_job.job_name
print(BASELINE_JOB)

## 2.3.3 Shadow variant vs production variant 📝

(Read this while the baseline job runs. Hands-on is in notebook 4.1a, where the endpoints exist.)

| | Shadow variant | A/B test (production variants) |
|---|---|---|
| Who gets the new model's answer? | Nobody: responses are logged, not returned | A share of real users |
| Risk to users | None | Some |
| What you compare | Latency, errors, and predictions on the *same* requests | Business outcome per variant |
| How | `ShadowProductionVariants` in the endpoint config (or a shadow test in the console) | Variant weights in `ProductionVariants` |

- A shadow variant receives a **copy** of a chosen percentage of live traffic.
- Compare with CloudWatch metrics per variant (`ModelLatency`, `Invocation5XXErrors`) and with **data capture** to compare predictions.
- Use shadow testing to validate a new model, container or instance type before moving traffic; then shift traffic with canary or linear deployment.

Wait for the baseline job.

In [ ]:
def processing_status(job_name):
    return sm.describe_processing_job(ProcessingJobName=job_name)["ProcessingJobStatus"]

U.wait_until(lambda: processing_status(BASELINE_JOB), ok=("Completed",), bad=("Failed", "Stopped"),
             delay=30, label="baseline job")

The job wrote `statistics.json` and `constraints.json`. Look at the statistics of the first features.

In [ ]:
stats = json.loads(U.read_bytes(f"{BASELINE_PREFIX}/statistics.json"))
pd.DataFrame([{"feature": f["name"], "type": f["inferred_type"],
               **{k: f.get("numerical_statistics", {}).get(k) for k in ("mean", "std_dev", "min", "max")}}
              for f in stats["features"]]).head(8)

And the suggested constraints: data type, completeness (share of non-missing values), sign.
A monitoring schedule (notebook 4.1a) reports a **violation** when live data breaks one of these.

In [ ]:
constraints = json.loads(U.read_bytes(f"{BASELINE_PREFIX}/constraints.json"))
print("monitoring config:", list(constraints.get("monitoring_config", {})))
pd.DataFrame(constraints["features"]).head(8)

Save the model quality baseline (test-set metrics) next to it, so later notebooks can compare against it.

In [ ]:
quality = {"roc_auc": float(roc_auc_score(y_true, y_score)), "pr_auc": float(average_precision_score(y_true, y_score)),
           "threshold": 0.5, "recall": float(recall_score(y_true, pred)), "precision": float(precision_score(y_true, pred))}
U.client("s3").put_object(Bucket=U.bucket(), Key=f"{BASELINE_PREFIX}/model_quality.json", Body=json.dumps(quality))
quality

## 2.3.4 Explain model outputs

- **Global** explanation: which features matter overall. **Local** explanation: why *this* claim got its score.
- **SHAP** values split one prediction into a contribution per feature (they add up to the score minus the average score).
- **SageMaker Clarify** computes SHAP for any model behind a SageMaker model/endpoint (Kernel SHAP), as a processing job.
- Also know: feature importance from tree models (fast, global only), partial dependence plots, and for FMs: citations and traces.

Pick the claims to explain: the 40 highest-scored test claims (what investigators will ask about) plus
60 random ones. Clarify wants a CSV; we give it the label first and no header.

In [ ]:
test_scored = test.assign(score=y_score)
sample = pd.concat([test_scored.nlargest(40, "score"), test_scored.sample(60, random_state=1)])
sample = sample[~sample.index.duplicated()]
SAMPLE_URI = U.write_df(sample[cols], f"{U.PREFIX}/clarify/input/sample.csv", header=False)
print(SAMPLE_URI, sample.shape, "mean score of the 40 flagged:", round(sample["score"].head(40).mean(), 3))

Configure the Clarify job: where the data is, which model to call, and the SHAP settings.
The SHAP **baseline** is the "average claim" (feature means): contributions are measured against it.

In [ ]:
from sagemaker import clarify
CLARIFY_OUT = f"{U.PREFIX}/clarify/output"
data_config = clarify.DataConfig(s3_data_input_path=SAMPLE_URI, s3_output_path=U.s3_uri(CLARIFY_OUT),
                                 label=F.LABEL, headers=cols, dataset_type="text/csv")
model_config = clarify.ModelConfig(model_name=EVAL_MODEL, instance_count=1, instance_type="ml.m5.large",
                                   accept_type="text/csv", content_type="text/csv")
shap_config = clarify.SHAPConfig(baseline=[train[F.FEATURES].mean().round(5).tolist()], num_samples=100,
                                 agg_method="mean_abs", save_local_shap_values=True, seed=42)
print("baseline row has", len(shap_config.shap_config["baseline"][0]), "features")

Start the explainability job (about 10 to 15 minutes). Clarify creates a temporary endpoint for the
model, calls it many times with perturbed rows, and deletes it at the end.

In [ ]:
clarify_job = clarify.SageMakerClarifyProcessor(
    role=U.role_arn(), instance_count=1, instance_type="ml.m5.xlarge", sagemaker_session=sm_session,
    tags=U.TAGS, job_name_prefix="safedrive-clarify")
clarify_job.run_explainability(data_config=data_config, model_config=model_config,
                               explainability_config=shap_config, wait=False, logs=False)
CLARIFY_JOB = clarify_job.latest_job.job_name
print(CLARIFY_JOB)

**While it runs: a quick local look.** Train a similar XGBoost model locally. Tree models can return
exact SHAP values directly (`pred_contribs=True`), which is the same idea Clarify applies to any model.

In [ ]:
local = xgb.XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.1, random_state=42).fit(Xtr, ytr)
contrib = local.get_booster().predict(xgb.DMatrix(sample[F.FEATURES]), pred_contribs=True)
shap_local = pd.DataFrame(contrib[:, :-1], columns=F.FEATURES, index=sample.index)   # last column = base value
shap_local.abs().mean().sort_values(ascending=False).head(8).round(3)

Local explanation of one flagged claim: the features that pushed its score up (+) or down (-, in log-odds units),
next to the feature values.

In [ ]:
row = sample.index[0]
one = pd.DataFrame({"value": sample.loc[row, F.FEATURES], "shap": shap_local.loc[row]})
print("model score:", round(sample.loc[row, "score"], 3))
one.reindex(one["shap"].abs().sort_values(ascending=False).index).head(6).round(3)

Wait for the Clarify job.

In [ ]:
U.wait_until(lambda: processing_status(CLARIFY_JOB), ok=("Completed",), bad=("Failed", "Stopped"),
             delay=30, timeout=2400, label="clarify job")

Read the global SHAP values from `analysis.json`. The folder also has a PDF/HTML report and the
per-claim SHAP values (`explanations_shap/out.csv`).

In [ ]:
analysis = json.loads(U.read_bytes(f"{CLARIFY_OUT}/analysis.json"))
kernel = analysis["explanations"]["kernel_shap"]
global_shap = pd.Series(next(iter(kernel.values()))["global_shap_values"]).sort_values(ascending=False)
global_shap.head(8).round(4)

Plot the top features. Do they match the local quick look and common sense (no police report, high
claim-to-value ratio, late reporting)?

In [ ]:
global_shap.head(10).iloc[::-1].plot.barh(figsize=(6, 3.2), title="Clarify: mean |SHAP| per feature")
plt.tight_layout(); plt.show()

## 2.3.5 Debug model convergence issues

| Symptom | Likely cause | Fix |
|---|---|---|
| Loss jumps around, grows, or becomes NaN | Learning rate too high, exploding gradients | Lower learning rate, gradient clipping, normalize inputs |
| Loss falls very slowly | Learning rate too low, vanishing gradients, poor initialization | Raise learning rate, learning-rate schedule, better init, batch norm |
| Training loss falls, validation loss rises | Overfitting | Regularization, early stopping, more data |
| Loss flat from the start | Data or label bug, features not scaled, dead ReLUs | Check the data pipeline first |

**SageMaker Debugger** saves tensors and metrics during training and runs **rules** on them in a
separate processing job. A rule can also **stop the training job** to save money.

Start a training job with three built-in Debugger rules. The hyperparameters are bad on purpose
(very deep trees, `eta=1`), so validation loss should get worse while training loss keeps falling.

In [ ]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.debugger import Rule, rule_configs, DebuggerHookConfig, CollectionConfig
rules = [Rule.sagemaker(rule_configs.loss_not_decreasing()), Rule.sagemaker(rule_configs.overfit()),
         Rule.sagemaker(rule_configs.overtraining())]
hook = DebuggerHookConfig(s3_output_path=U.s3_uri(f"{U.PREFIX}/debugger"),
                          collection_configs=[CollectionConfig(name="metrics", parameters={"save_interval": "5"})])
dbg = Estimator(image_uri=U.get_state("xgb_image"), role=U.role_arn(), instance_count=1,
                instance_type="ml.m5.large", output_path=U.s3_uri(f"{U.PREFIX}/models"),
                sagemaker_session=sm_session, base_job_name="safedrive-fraud-debug", tags=U.TAGS,
                rules=rules, debugger_hook_config=hook, disable_profiler=True)
dbg.set_hyperparameters(objective="binary:logistic", eval_metric="logloss", num_round=120, max_depth=12, eta=1.0)
[r.name for r in rules]

In [ ]:
dbg.fit({"train": TrainingInput(state["train_s3"], content_type="text/csv"),
         "validation": TrainingInput(state["validation_s3"], content_type="text/csv")}, wait=False)
DEBUG_JOB = dbg.latest_training_job.name
print(DEBUG_JOB)

**While it runs: see the learning-rate effect locally.** Plain gradient descent for logistic regression
with three learning rates. Too low barely moves, a good value converges, too high jumps around.

In [ ]:
from sklearn.preprocessing import StandardScaler
Xs = np.c_[np.ones(len(Xtr)), StandardScaler().fit_transform(Xtr)]
y = ytr.values

def descend(lr, steps=40):
    w, losses = np.zeros(Xs.shape[1]), []
    for _ in range(steps):
        p = 1 / (1 + np.exp(-np.clip(Xs @ w, -30, 30)))
        losses.append(-np.mean(y * np.log(p + 1e-9) + (1 - y) * np.log(1 - p + 1e-9)))
        w -= lr * Xs.T @ (p - y) / len(y)
    return losses

curves = {lr: descend(lr) for lr in (0.001, 0.5, 30)}
pd.DataFrame(curves).iloc[::8].round(3)

Plot the three loss curves.

In [ ]:
plt.figure(figsize=(6, 3))
for lr, losses in curves.items():
    plt.plot(losses, label=f"learning rate {lr}")
plt.xlabel("step"); plt.ylabel("training loss"); plt.legend(); plt.title("Learning rate and convergence")
plt.show()

Wait for the training job and then for the rule evaluations to finish.

In [ ]:
U.wait_until(lambda: sm.describe_training_job(TrainingJobName=DEBUG_JOB)["TrainingJobStatus"],
             ok=("Completed", "Stopped"), bad=("Failed",), delay=30, label="training job")

In [ ]:
def rule_statuses():
    d = sm.describe_training_job(TrainingJobName=DEBUG_JOB)
    return [(r["RuleConfigurationName"], r["RuleEvaluationStatus"]) for r in d.get("DebugRuleEvaluationStatuses", [])]

U.wait_until(lambda: "running" if any(s == "InProgress" for _, s in rule_statuses()) else "done",
             ok=("done",), delay=30, label="debugger rules")

Rule results: `IssuesFound` means the rule fired, `NoIssuesFound` means the run looked healthy for that rule.

In [ ]:
d = sm.describe_training_job(TrainingJobName=DEBUG_JOB)
pd.DataFrame(d["DebugRuleEvaluationStatuses"])[["RuleConfigurationName", "RuleEvaluationStatus", "StatusDetails"]]

The metric history of the job confirms it: compare the final training and validation loss.

In [ ]:
pd.DataFrame(d["FinalMetricDataList"])[["MetricName", "Value"]]

## Exam traps

- High accuracy on imbalanced data proves nothing. Look for recall, precision, F1 or PR AUC in the answer options.
- The **model quality** monitor needs ground truth labels merged with captured predictions; the **data quality** monitor does not.
- A shadow variant's responses are never returned to the caller. If users must see the new model, it is an A/B test.
- Clarify explains (SHAP) and measures bias; Debugger watches *training*; Model Monitor watches *production*. The exam mixes these three up.
- Lowering the classification threshold raises recall and lowers precision, never both up.

## Quiz

**Q1.** A fraud model has 98% accuracy but catches only 10% of fraud cases; fraud is 2% of the data.
Which metric should the team optimize to catch more fraud?
A) Accuracy  B) Recall  C) Specificity  D) R²

<details><summary>Answer</summary>B. Recall measures the share of real fraud that is caught. Accuracy is dominated by the legitimate class.</details>

**Q2.** A team wants to track the hyperparameters, metrics and model files of every training run and compare
runs in a UI, using open-source tooling with little operational work. What should they use?
A) CloudTrail  B) A SageMaker AI managed MLflow tracking server  C) SageMaker Debugger  D) AWS Config

<details><summary>Answer</summary>B. MLflow tracking records runs; SageMaker hosts and manages the server.</details>

**Q3.** A lender must show a customer which features drove the decision on *their* loan application.
Which capability provides this?
A) SageMaker Clarify SHAP local explanations  B) Model Monitor data quality  C) Debugger `overfit` rule  D) ROC curve

<details><summary>Answer</summary>A. SHAP values give a per-prediction contribution for each feature.</details>

**Q4.** During training of a deep network the loss becomes NaN after a few hundred steps. Which Debugger
rule and fix match?
A) `overfit`; add dropout  B) `exploding_tensor`; lower the learning rate or clip gradients
C) `class_imbalance`; resample  D) `dead_relu`; raise the learning rate

<details><summary>Answer</summary>B. NaN loss comes from exploding values, typically a learning rate that is too high.</details>

**Q5.** A team wants to test a new model version on real production requests with zero risk to users before
shifting any traffic. What should they deploy?
A) An A/B test with 50/50 weights  B) A shadow variant  C) A blue/green all-at-once deployment  D) A batch transform job

<details><summary>Answer</summary>B. The shadow variant gets a copy of live traffic; its responses are only logged.</details>

## Save state
`monitor_baseline_prefix` is the S3 URI of the folder with `statistics.json` and `constraints.json`
(used by the monitoring schedule in notebook 4.1a).

In [ ]:
state = U.save_state(monitor_baseline_prefix=U.s3_uri(BASELINE_PREFIX),
                     fraud_test_metrics=quality, clarify_output=U.s3_uri(CLARIFY_OUT))
{k: state[k] for k in ("monitor_baseline_prefix", "fraud_test_metrics")}

## Cleanup
Jobs have ended and cost nothing more. Delete the two resources that remain: the MLflow tracking
server (hourly cost) and the evaluation model object.
**Kept for later notebooks:** the baseline files in `s3://<bucket>/safedrive/monitoring/baseline/` and the Clarify report.

In [ ]:
U.safe(sm.delete_mlflow_tracking_server, TrackingServerName=MLFLOW_NAME)

In [ ]:
U.safe(sm.delete_model, ModelName=EVAL_MODEL)